# MY FILMS & BOOKS LOG

## Imports & Directory Setup

In [5]:
import os
import json
import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MultiLabelBinarizer

# Define File Paths based on your directory structure
LB_PERSONAL_DIR = '../Data/letterboxd/letterboxd_personal_stats'
LB_PUBLIC_DIR = '../Data/letterboxd/letterboxd_public_stats'

SG_PERSONAL_DIR = '../Data/storygraph/storygraph_personal_stats'
SG_PUBLIC_DIR = '../Data/storygraph/storygraph_public_stats'

# Ensure output public directories exist
os.makedirs(LB_PUBLIC_DIR, exist_ok=True)
os.makedirs(SG_PUBLIC_DIR, exist_ok=True)

print("Directories verified and ready.")

Directories verified and ready.


## Load Personal Data (CSV Files)

### Letterboxd data

In [6]:
# Load Letterboxd data
ratings_path = os.path.join(LB_PERSONAL_DIR, 'ratings.csv')
watchlist_path = os.path.join(LB_PERSONAL_DIR, 'watchlist.csv')

ratings_df = pd.read_csv(ratings_path)
watchlist_df = pd.read_csv(watchlist_path)

ratings_df['type'] = 'movie'
ratings_df['status'] = 'watched'
watchlist_df['type'] = 'movie'
watchlist_df['status'] = 'watchlist'
watchlist_df['Rating'] = np.nan

movies_df = pd.concat([ratings_df, watchlist_df], ignore_index=True)

movies_df.head()


,Date,Name,Year,Letterboxd URI,Rating,type,status
0,2023-03-04,Pulp Fiction,1994,https://boxd.it/29Pq,5.0,movie,watched
1,2023-03-04,Parasite,2019,https://boxd.it/hTha,5.0,movie,watched
2,2023-03-04,Knives Out,2019,https://boxd.it/jWEA,5.0,movie,watched
3,2023-03-04,Spider-Man: Into the Spider-Verse,2018,https://boxd.it/azpY,5.0,movie,watched
4,2023-03-04,Whiplash,2014,https://boxd.it/7bQA,5.0,movie,watched


### StoryGraph data

In [7]:
# Load StoryGraph data
reading_path = os.path.join(SG_PERSONAL_DIR, 'reading_log.csv')
reading_df = pd.read_csv(reading_path)

books_df = pd.DataFrame({
    'Title': reading_df['Title'],
    'Author': reading_df['Authors'],
    'Rating': pd.to_numeric(reading_df['Star Rating'], errors='coerce'),
    'status': reading_df['Read Status'].map({'read': 'read', 'to-read': 'tbr'}),
    'type': 'book',
    'Moods': reading_df['Moods'].fillna(''),
    'Pace': reading_df['Pace'].fillna('')
})
books_df = books_df[books_df['status'].isin(['read', 'tbr'])].reset_index(drop=True)

books_df.head()

,Title,Author,Rating,status,type,Moods,Pace
0,Who's Afraid of Virginia Woolf?,Edward Albee,NaN,tbr,book,,
1,Franny and Zooey,J.D. Salinger,4.25,read,book,,
2,Parable of the Sower,Octavia E. Butler,NaN,tbr,book,,
3,Erotic Stories for Punjabi Widows,Balli Kaur Jaswal,3.75,read,book,,
4,Consider the Lobster And Other Essays,David Foster Wallace,NaN,tbr,book,,


In [9]:
print(f"Loaded {len(movies_df)} total movies ({len(ratings_df)} rated, {len(watchlist_df)} watchlist).")

read_count = (books_df['status'] == 'read').sum()
tbr_count = (books_df['status'] == 'tbr').sum()
print(f"Loaded {len(books_df)} total books ({read_count} read, {tbr_count} to-be-read).")

Loaded 445 total movies (208 rated, 237 watchlist).
Loaded 161 total books (28 read, 133 to-be-read).


## Movies and Books Data Scraping

### Letterboxd public data

In [ ]:
"""
Scrapes PUBLIC metadata for every film / book in your exports.
  Letterboxd -> public_rating, rating_count, genres, themes   (no login needed)
  StoryGraph -> public_rating, moods, pace                    (needs your own session cookie)

Usage (from your notebook or terminal):
    from scrape_metadata import scrape_letterboxd, scrape_storygraph
    lb = scrape_letterboxd(movies_df, LB_PUBLIC_DIR)
    sg = scrape_storygraph(books_df, SG_PUBLIC_DIR)

Results are cached row-by-row, so if it stops you can just re-run it.
Raw HTML of the first few pages is saved to <dir>/debug_html/ so you can
inspect what the site really returned if a field comes back empty.
"""
import json, os, re, time, random
import requests
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}

def _sleep(lo=1.0, hi=2.0):
    time.sleep(random.uniform(lo, hi))          # be polite; avoids rate limiting

def _dedupe(items):
    return list(dict.fromkeys(i for i in items if i))

# ───────────────────────── Letterboxd ─────────────────────────
def parse_letterboxd(html):
    soup = BeautifulSoup(html, "html.parser")
    out = {"public_rating": None, "rating_count": None, "poster_url": None, "genres": [], "themes": []}

    # 1) Rating + genres from the JSON-LD block (most stable source)
    for tag in soup.find_all("script", type="application/ld+json"):
        raw = re.sub(r"/\*.*?\*/", "", tag.string or "", flags=re.S).strip()
        try:
            data = json.loads(raw)
        except json.JSONDecodeError:
            continue
        agg = data.get("aggregateRating") or {}
        if agg.get("ratingValue") is not None:
            out["public_rating"] = float(agg["ratingValue"])
            out["rating_count"] = agg.get("ratingCount")
        img = data.get("image")
        if img:
            out["poster_url"] = img.split("?")[0]   # poster image (we resize it later)
        g = data.get("genre")
        if g:
            out["genres"] = [g] if isinstance(g, str) else list(g)

    # 2) Fallback rating: twitter meta tag ("4.23 out of 5")
    if out["public_rating"] is None:
        m = soup.find("meta", attrs={"name": "twitter:data2"})
        if m:
            try: out["public_rating"] = float(m["content"].split()[0])
            except (ValueError, KeyError, IndexError): pass

    # 3) Genres & themes: select by URL pattern, not by container/heading,
    #    so layout changes don't break it.
    def links(prefixes):
        return [a.get_text(" ", strip=True) for a in soup.find_all("a", href=True)
                if any(a["href"].startswith(p) for p in prefixes)]
    if not out["genres"]:
        out["genres"] = _dedupe(links(["/films/genre/"]))
    out["themes"] = _dedupe(links(["/films/theme/", "/films/mini-theme/", "/films/nanogenre/"]))
    # drop Letterboxd's "Show All…" style links
    out["themes"] = [t for t in out["themes"] if not t.lower().startswith("show all")]
    out["genres"] = _dedupe(out["genres"])
    return out

def scrape_letterboxd(df, out_dir, cache_name="scraped_letterboxd_stats.json", debug_n=3):
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, cache_name)
    cache = {}
    if os.path.exists(path):
        cache = {r["Letterboxd URI"]: r for r in json.load(open(path))
                 if r.get("metadata_status") == "ok" and r.get("poster_url")}
    dbg = os.path.join(out_dir, "debug_html"); os.makedirs(dbg, exist_ok=True)
    session = requests.Session(); session.headers.update(HEADERS)
    rows = []
    for i, (_, r) in enumerate(df.iterrows()):
        uri = r["Letterboxd URI"]
        if uri in cache:
            rows.append(cache[uri]); continue
        rec = {"Name": r["Name"], "Year": int(r["Year"]), "Letterboxd URI": uri,
               "public_rating": None, "rating_count": None, "poster_url": None, "genres": [], "themes": []}
        try:
            resp = session.get(uri, timeout=20, allow_redirects=True)
            resp.raise_for_status()
            if i < debug_n:
                open(os.path.join(dbg, f"lb_{i}.html"), "w", encoding="utf-8").write(resp.text)
            rec.update(parse_letterboxd(resp.text))
            missing = [k for k in ("public_rating", "poster_url", "genres", "themes") if not rec[k]]
            rec["metadata_status"] = "ok" if not missing else "missing: " + ", ".join(missing)
            rec["resolved_url"] = resp.url
        except requests.RequestException as e:
            rec["metadata_status"] = f"request_error: {e}"
        rows.append(rec)
        if (i + 1) % 25 == 0:
            json.dump(rows, open(path, "w"), indent=2); print(f"  Letterboxd {i+1}/{len(df)}")
        _sleep()
    json.dump(rows, open(path, "w"), indent=2)
    return rows

# ── Run Letterboxd scrape (safe to re-run; finished rows are cached) ──
lb_rows = scrape_letterboxd(movies_df, LB_PUBLIC_DIR)
print(pd.Series([r["metadata_status"] for r in lb_rows]).value_counts())


### StoryGraph public data
StoryGraph blocks scripts, so rating / genres / moods / pace live in `scraped_storygraph_stats.json` as hand-entered data.

In [ ]:
# StoryGraph sits behind a Cloudflare bot check, so the public rating / genres / moods / pace
# in this file were entered by hand. Here we just load and sanity-check them.
sg_path = os.path.join(SG_PUBLIC_DIR, 'scraped_storygraph_stats.json')
with open(sg_path, encoding='utf-8') as f:
    sg_rows = json.load(f)

sg_df = pd.DataFrame(sg_rows)
print(f"{len(sg_df)} books | rating: {sg_df['public_rating'].notna().sum()} | "
      f"genres: {(sg_df['genres'].str.len() > 0).sum()} | moods: {(sg_df['moods'].str.len() > 0).sum()} | "
      f"pace: {sg_df['pace'].notna().sum()}")

problems = sg_df[
    sg_df['public_rating'].isna() | ~sg_df['public_rating'].between(0, 5) |
    (sg_df['pace'].notna() & ~sg_df['pace'].isin(['slow', 'medium', 'fast']))
]
print("Rows to double-check:" if len(problems) else "All rows look valid.")
if len(problems):
    display(problems[['Title', 'public_rating', 'pace']])

## Posters & Covers
Posters come from Letterboxd, book covers from Open Library (Google Books as a fallback). Images are saved next to the sketch, and the file names are written back into the public-stats JSON files.

In [ ]:
# Download movie posters (from Letterboxd) and book covers (Open Library, then Google Books)
# into the sketch's data folder, and record the local file names in the public-stats JSON files.
import re, time

IMG_DIR = '../Books_N_Movies/data/images'
POSTER_DIR = os.path.join(IMG_DIR, 'posters')
COVER_DIR = os.path.join(IMG_DIR, 'covers')
os.makedirs(POSTER_DIR, exist_ok=True)
os.makedirs(COVER_DIR, exist_ok=True)

def boxd_id(uri):
    return uri.rstrip('/').split('/')[-1]

def try_download(url, dest, min_bytes=2000):
    if os.path.exists(dest):
        return True
    try:
        r = requests.get(url, headers=HEADERS, timeout=20)
    except requests.RequestException:
        return False
    if r.status_code == 200 and len(r.content) >= min_bytes and r.headers.get('content-type', '').startswith('image'):
        with open(dest, 'wb') as f:
            f.write(r.content)
        return True
    return False

# ---- Movie posters (small 150x225 version keeps the sketch light) ----
lb_json_path = os.path.join(LB_PUBLIC_DIR, 'scraped_letterboxd_stats.json')
with open(lb_json_path, encoding='utf-8') as f:
    lb_rows = json.load(f)

for i, row in enumerate(lb_rows):
    fname = f"m_{boxd_id(row['Letterboxd URI'])}.jpg"
    url = row.get('poster_url')
    url = re.sub(r'-0-\d+-0-\d+-crop', '-0-150-0-225-crop', url) if url else None
    dest = os.path.join(POSTER_DIR, fname)
    existed = os.path.exists(dest)
    ok = bool(url) and try_download(url, dest)
    row['poster_file'] = f'data/images/posters/{fname}' if ok else None
    if ok and not existed:
        time.sleep(0.05)   # be polite to the image server
with open(lb_json_path, 'w', encoding='utf-8') as f:
    json.dump(lb_rows, f, indent=2, ensure_ascii=False)
print(f"Posters: {sum(r['poster_file'] is not None for r in lb_rows)}/{len(lb_rows)}")

# ---- Book covers: Open Library by ISBN -> Google Books -> Open Library title search ----
def cover_candidates(row):
    isbn = str(row.get('ISBN') or '')
    if isbn.isdigit():
        yield f'https://covers.openlibrary.org/b/isbn/{isbn}-M.jpg?default=false'
    queries = ([f'isbn:{isbn}'] if isbn.isdigit() else []) + [f"intitle:{row['Title']} inauthor:{row['Author']}"]
    for q in queries:
        try:
            j = requests.get('https://www.googleapis.com/books/v1/volumes',
                             params={'q': q, 'maxResults': 1}, timeout=20).json()
        except (requests.RequestException, ValueError):
            continue
        for item in j.get('items', [])[:1]:
            links = item.get('volumeInfo', {}).get('imageLinks', {})
            u = links.get('thumbnail') or links.get('smallThumbnail')
            if u:
                yield u.replace('http://', 'https://').replace('&edge=curl', '')
        time.sleep(0.3)
    try:
        j = requests.get('https://openlibrary.org/search.json',
                         params={'title': row['Title'], 'author': row['Author'], 'limit': 1, 'fields': 'cover_i'},
                         timeout=20).json()
        cid = (j.get('docs') or [{}])[0].get('cover_i')
        if cid:
            yield f'https://covers.openlibrary.org/b/id/{cid}-M.jpg?default=false'
    except (requests.RequestException, ValueError):
        pass

for row in sg_rows:
    isbn = str(row.get('ISBN') or '')
    key = isbn if isbn.isdigit() else re.sub(r'\W+', '-', row['Title'].lower()).strip('-')
    fname = f'b_{key}.jpg'
    dest = os.path.join(COVER_DIR, fname)
    ok = os.path.exists(dest)
    if not ok:
        for u in cover_candidates(row):
            if try_download(u, dest):
                ok = True
                row['cover_url'] = u
                break
    row['cover_file'] = f'data/images/covers/{fname}' if ok else None
with open(sg_path, 'w', encoding='utf-8') as f:
    json.dump(sg_rows, f, indent=2, ensure_ascii=False)
print(f"Covers: {sum(r['cover_file'] is not None for r in sg_rows)}/{len(sg_rows)}")
print("No cover found for:", [r['Title'] for r in sg_rows if r['cover_file'] is None])

## Build the modelling dataset
One list of films + books, each with its public rating and feature terms (genres, themes, moods, pace, decade).

In [ ]:
# Build one list of "items" (films + books) with the features the model will use.
from collections import Counter

def boxd_id(uri):
    return uri.rstrip('/').split('/')[-1]

# the public-stats JSON (with poster file names) is the source of truth for movie features
with open(os.path.join(LB_PUBLIC_DIR, 'scraped_letterboxd_stats.json'), encoding='utf-8') as f:
    lb_by_uri = {r['Letterboxd URI']: r for r in json.load(f)}
with open(os.path.join(SG_PUBLIC_DIR, 'scraped_storygraph_stats.json'), encoding='utf-8') as f:
    sg_by_key = {(r['Title'], r['Author']): r for r in json.load(f)}

movie_items = []
for _, r in movies_df.iterrows():
    m = lb_by_uri[r['Letterboxd URI']]
    movie_items.append(dict(
        id='m_' + boxd_id(r['Letterboxd URI']), type='movie',
        title=r['Name'], subtitle=str(int(r['Year'])),
        rating=None if pd.isna(r['Rating']) else float(r['Rating']),
        public_rating=m['public_rating'],
        genres=m['genres'], themes=m['themes'], moods=[], pace=None,
        # model terms: g: genre, t: theme, d: decade
        terms=['g:' + g.lower() for g in m['genres']] + ['t:' + t.lower() for t in m['themes']]
              + ['d:' + str(int(r['Year']) // 10 * 10) + 's'],
        image=m.get('poster_file'), url=r['Letterboxd URI']))

book_items = []
for _, r in books_df.iterrows():
    b = sg_by_key[(r['Title'], r['Author'])]
    if r['status'] == 'read' and pd.isna(r['Rating']):
        continue                                  # read but never rated: can't train on it, nothing to predict
    book_items.append(dict(
        id='b_' + (b['ISBN'] if str(b['ISBN']).isdigit() else re.sub(r'\W+', '-', r['Title'].lower()).strip('-')),
        type='book', title=r['Title'], subtitle=r['Author'],
        rating=None if pd.isna(r['Rating']) else float(r['Rating']),
        public_rating=b['public_rating'],
        genres=b['genres'], themes=[], moods=b['moods'], pace=b['pace'],
        # model terms: g: genre, m: mood, p: pace
        terms=['g:' + g.lower() for g in b['genres']] + ['m:' + m.lower() for m in b['moods']]
              + (['p:' + b['pace']] if b['pace'] else []),
        image=b.get('cover_file'), url=None))

for name, items in [('movies', movie_items), ('books', book_items)]:
    rated = sum(i['rating'] is not None for i in items)
    print(f"{name}: {len(items)} items, {rated} rated, {len(items) - rated} to predict")

## Predictive model: which algorithm?
Compares a baseline, public-rating-only, **ridge regression**, k-nearest neighbours and a random forest using repeated 60/20/20 train/validation/test splits.

In [ ]:
# ───────────── Predictive model ─────────────
# Goal: predict MY rating of something I haven't seen/read yet.
#   features : public rating (z-scored) + one-hot genres / themes / moods / pace / decade
#   model    : ridge regression  (linear, L2-shrunk toward the average rating)
# Why ridge?  Only ~200 rated films and ~28 rated books against hundreds of one-hot features:
#   plain regression would overfit, trees/kNN need far more data, and k-means is unsupervised.
#   Ridge is stable on tiny data, readable (each genre/mood gets a weight), and re-fits in
#   milliseconds so the browser can re-train it live when you enter a new rating.
# Logistic regression is for classes; a 3.5 isn't "wrong" when the truth is 4, so we regress.
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error

MIN_DF = 3                                   # ignore terms that appear in fewer than 3 items

def build_features(items, min_df=MIN_DF):
    counts = Counter(t for it in items for t in set(it['terms']))
    vocab = sorted(t for t, c in counts.items() if c >= min_df)
    pubs = np.array([np.nan if it['public_rating'] is None else it['public_rating'] for it in items], float)
    pubs = np.where(np.isnan(pubs), np.nanmean(pubs), pubs)
    mu, sd = pubs.mean(), pubs.std() or 1.0
    idx = {t: i + 1 for i, t in enumerate(vocab)}
    X = np.zeros((len(items), 1 + len(vocab)))
    X[:, 0] = (pubs - mu) / sd
    for i, it in enumerate(items):
        for t in set(it['terms']):
            if t in idx:
                X[i, idx[t]] = 1.0
    return X, vocab, mu, sd

def ridge_fit(X, y, alpha, num_scale=1.0):
    X = X.copy(); X[:, 0] *= num_scale
    xm, ym = X.mean(0), y.mean()
    Xc = X - xm
    w = np.linalg.solve(Xc.T @ Xc + alpha * np.eye(X.shape[1]), Xc.T @ (y - ym))
    return dict(w=w, xm=xm, ym=ym, s=num_scale)

def ridge_predict(model, X):
    X = X.copy(); X[:, 0] *= model['s']
    return np.clip(model['ym'] + (X - model['xm']) @ model['w'], 0.5, 5.0)

GRID = [(a, s) for a in (1, 3, 10, 30, 100) for s in (1, 3)]      # (alpha, public-rating scale)

def one_split(X, y, seed):
    """60% train / 20% validation / 20% test. Tune on validation, report on test."""
    idx = np.arange(len(y))
    tr, rest = train_test_split(idx, test_size=0.4, random_state=seed)
    va, te = train_test_split(rest, test_size=0.5, random_state=seed)
    mae = lambda yt, yp: mean_absolute_error(yt, yp)
    out = {'val': {}, 'test': {}}
    # ridge grid
    for (a, s) in GRID:
        m = ridge_fit(X[tr], y[tr], a, s)
        out['val'][(a, s)] = mae(y[va], ridge_predict(m, X[va]))
        out['test'][(a, s)] = mae(y[te], ridge_predict(m, X[te]))
    best = min(GRID, key=lambda g: out['val'][g])
    res = {'ridge (tuned)': out['test'][best]}
    res['mean rating (baseline)'] = mae(y[te], np.full(len(te), y[tr].mean()))
    m = ridge_fit(X[tr][:, :1], y[tr], 0.0)
    res['public rating only'] = mae(y[te], ridge_predict(m, X[te][:, :1]))
    # kNN (cosine), k tuned on validation
    kbest = min((3, 5, 8, 12), key=lambda k: mae(y[va], KNeighborsRegressor(min(k, len(tr)), metric='cosine', weights='distance')
                .fit(X[tr], y[tr]).predict(X[va])))
    res['k-nearest neighbours'] = mae(y[te], KNeighborsRegressor(min(kbest, len(tr)), metric='cosine', weights='distance')
                .fit(X[tr], y[tr]).predict(X[te]))
    rf = RandomForestRegressor(150, min_samples_leaf=2, random_state=0).fit(X[tr], y[tr])
    res['random forest'] = mae(y[te], rf.predict(X[te]))
    return out['val'], res

def evaluate(items, n_splits=20):
    X, vocab, mu, sd = build_features(items)
    known = np.array([it['rating'] is not None for it in items])
    Xk, yk = X[known], np.array([it['rating'] for it in items if it['rating'] is not None])
    vals, tests = [], []
    for seed in range(n_splits):
        v, t = one_split(Xk, yk, seed)
        vals.append(v); tests.append(t)
    mean_val = {g: np.mean([v[g] for v in vals]) for g in GRID}
    best = min(GRID, key=lambda g: mean_val[g])
    table = pd.DataFrame(tests).agg(['mean', 'std']).T.round(3)
    return dict(X=X, vocab=vocab, mu=mu, sd=sd, known=known, y=yk, best=best, table=table, n_known=len(yk))

print("Comparing models: mean test error (stars) over 20 random 60/20/20 splits")
movie_eval = evaluate(movie_items)
book_eval = evaluate(book_items)
cmp = pd.concat({'movies': movie_eval['table'], 'books': book_eval['table']}, axis=1)
display(cmp)
print(f"Chosen ridge settings (alpha, public-rating weight): movies {movie_eval['best']}, books {book_eval['best']}")

## Fit on all ratings, predict the unrated, export for p5
Writes `Books_N_Movies/data/viz_data.json`. The sketch re-trains the same ridge model live when you enter a rating.

In [ ]:
# Fit the chosen model on ALL ratings, predict the unrated items, and export everything the p5 sketch needs.
# The sketch re-implements the same ridge fit in JavaScript so predictions update live when you enter a rating.
def fit_and_predict(items, ev):
    alpha, scale = ev['best']
    model = ridge_fit(ev['X'][ev['known']], ev['y'], alpha, scale)
    preds = ridge_predict(model, ev['X'])
    for it, p in zip(items, preds):
        it['pred'] = None if it['rating'] is not None else round(float(p), 3)
    return model

movie_model = fit_and_predict(movie_items, movie_eval)
book_model = fit_and_predict(book_items, book_eval)

def top_terms(model, ev, n=6):
    w = pd.Series(model['w'][1:] * 1.0, index=ev['vocab'])
    return {'likes': w.nlargest(n).round(2).to_dict(), 'dislikes': w.nsmallest(n).round(2).to_dict(),
            'public_rating_weight': round(float(model['w'][0] * model['s']), 3)}

print("What the movie model learned about my taste:"); print(top_terms(movie_model, movie_eval))
print("What the book model learned about my taste:"); print(top_terms(book_model, book_eval))

def config(ev, items, step):
    alpha, scale = ev['best']
    t = ev['table']
    return dict(alpha=alpha, num_scale=scale, min_df=MIN_DF, pub_mean=round(float(ev['mu']), 5),
                pub_std=round(float(ev['sd']), 5), step=step, n_rated=int(ev['n_known']),
                mae_model=float(t.loc['ridge (tuned)', 'mean']), mae_baseline=float(t.loc['mean rating (baseline)', 'mean']),
                mae_crowd=float(t.loc['public rating only', 'mean']))

OUT_DIR = '../Books_N_Movies/data'
os.makedirs(OUT_DIR, exist_ok=True)
payload = {'config': {'movie': config(movie_eval, movie_items, 0.5), 'book': config(book_eval, book_items, 0.25)},
           'taste': {'movie': top_terms(movie_model, movie_eval), 'book': top_terms(book_model, book_eval)},
           'items': movie_items + book_items}
with open(os.path.join(OUT_DIR, 'viz_data.json'), 'w', encoding='utf-8') as f:
    json.dump(payload, f, ensure_ascii=False, indent=1)
print(f"Exported {len(payload['items'])} items to {OUT_DIR}/viz_data.json")